# Paderborn Bearing Fault Classification — Standard CNN

**Key fixes over the original notebook:**
1. **Data leakage fixed**: Raw signals are split into train/val/test *before* segmentation, so no segments from the same recording appear across splits.
2. **Residual blocks removed**: Pure standard CNN architecture only.
3. All other hyperparameters, architecture, and evaluation remain unchanged.

In [ ]:
import numpy as np
import pandas as pd
import os
import glob
from tqdm.notebook import tqdm
from scipy.io import loadmat

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────

INPUT_PATH   = '/kaggle/input/datasets/dippatel03/paderborn-db'
WINDOW_SIZE  = 1024   # samples per segment (1024 @ 64kHz ≈ 16 ms)
STRIDE       = 512    # 50% overlap → doubles dataset size
FILES_PER_FOLDER = 20 # default

BATCH_SIZE   = 64
EPOCHS       = 100    # EarlyStopping will cut this short
LEARNING_RATE = 1e-3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("Configuration set.")

## 1. Load Raw Signals (one signal per file)

In [ ]:
def extract_paderborn_signal(mat_content):
    """
    Extracts the vibration signal from Paderborn .mat files.
    Structure: mat['FileName'][0,0]['Y'][0,0]['Data']
    Falls back to largest array if structure differs.
    """
    for key in mat_content.keys():
        if key.startswith('__'):
            continue
        try:
            data = mat_content[key][0, 0]['Y'][0, 0]['Data']
            return np.array(data).flatten().astype(np.float32)
        except (IndexError, KeyError, TypeError):
            data_array = np.array(mat_content[key])
            return data_array.flatten().astype(np.float32)
    return None


def load_paderborn_dataset(base_path, files_per_folder=20):
    """
    Loads ALL .mat files from every subfolder.
    Each subfolder = one bearing class label.
    """
    all_signals = []
    all_labels  = []

    folders = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ])

    print(f"Found {len(folders)} class folders: {folders}\n")

    for folder in tqdm(folders, desc="Loading classes"):
        folder_path = os.path.join(base_path, folder)
        files = sorted(glob.glob(os.path.join(folder_path, "*.mat")))
        files_per_folder = len(files)

        loaded = 0
        for file_path in files[:files_per_folder]:
            try:
                mat_data = loadmat(file_path)
                signal   = extract_paderborn_signal(mat_data)
                if signal is not None and len(signal) > WINDOW_SIZE:
                    all_signals.append(signal)
                    all_labels.append(folder)
                    loaded += 1
            except Exception as e:
                print(f"  [WARN] {os.path.basename(file_path)}: {e}")

    return pd.DataFrame({'signal': all_signals, 'label': all_labels})


print("Starting data load...")
df = load_paderborn_dataset(INPUT_PATH, files_per_folder=FILES_PER_FOLDER)

print(f"\nTotal files loaded : {len(df)}")
print(f"Classes found      : {sorted(df['label'].unique())}")
print(f"Samples per class  :\n{df['label'].value_counts().sort_index()}")

In [ ]:
print(f"Example signal shape: {df['signal'][0].shape}")

In [ ]:
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

NUM_CLASSES = len(label_encoder.classes_)
print(f"Number of classes : {NUM_CLASSES}")
print(f"Class mapping     :")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i:2d} → {cls}")

## 2. Split Raw Signals BEFORE Segmentation (Prevents Data Leakage)

The critical fix: we split at the **file/recording level** first, then segment each split independently.  
This ensures no segments from the same recording appear in both train and test sets.

In [ ]:
# ── Split raw signals (file-level) ─────────────────────────────────
# 70% train  |  15% val  |  15% test

df_train_val, df_test = train_test_split(
    df,
    test_size=0.15,
    stratify=df['label'],
    random_state=RANDOM_SEED
)

df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.176,   # 0.176 × 0.85 ≈ 0.15 of total
    stratify=df_train_val['label'],
    random_state=RANDOM_SEED
)

print(f"Raw signal split (file-level):")
print(f"  Train files : {len(df_train)}")
print(f"  Val files   : {len(df_val)}")
print(f"  Test files  : {len(df_test)}")
print(f"  Total       : {len(df_train) + len(df_val) + len(df_test)}")

## 3. Segment Each Split Independently

In [ ]:
def segment_signals(df_split, window_size=WINDOW_SIZE, stride=STRIDE):
    """
    Splits each long signal into overlapping windows.
    Each segment is Z-score normalised independently.
    Returns:
        X : np.ndarray of shape (N, window_size)
        y : np.ndarray of shape (N,) with integer labels
    """
    X_list = []
    y_list = []

    for _, row in df_split.iterrows():
        signal = row['signal']
        label  = row['label_encoded']

        for start in range(0, len(signal) - window_size, stride):
            end     = start + window_size
            segment = signal[start:end]

            # Per-segment Z-score normalisation
            mu  = segment.mean()
            std = segment.std() + 1e-8
            segment = (segment - mu) / std

            X_list.append(segment)
            y_list.append(label)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)


# Segment each split independently — NO data leakage!
print(f"Segmenting with window={WINDOW_SIZE}, stride={STRIDE}...")

X_train, y_train = segment_signals(df_train)
print(f"  Train segments : {X_train.shape[0]}")

X_val, y_val = segment_signals(df_val)
print(f"  Val segments   : {X_val.shape[0]}")

X_test, y_test = segment_signals(df_test)
print(f"  Test segments  : {X_test.shape[0]}")

print(f"\nTotal segments   : {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]}")

# Print class distribution per split
for name, y_arr in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(y_arr, return_counts=True)
    print(f"\n{name} segments per class:")
    for u, c in zip(unique, counts):
        print(f"  {label_encoder.classes_[u]}: {c}")

In [ ]:
# Reshape for Keras: (N, time_steps, channels)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val   = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

# One-hot encode labels for Keras
y_train_oh = to_categorical(y_train, NUM_CLASSES)
y_val_oh   = to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = to_categorical(y_test,  NUM_CLASSES)

In [ ]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train
)
class_weights_dict = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights computed:")
for i, w in class_weights_dict.items():
    print(f"  Class {i:2d} ({label_encoder.classes_[i]}): {w:.3f}")

## 4. Build Standard CNN Model (No Residual Blocks)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# BUILDING BLOCKS
# ──────────────────────────────────────────────────────────────────

def conv_bn_relu(x, filters, kernel_size, strides=1):
    """Conv → BatchNorm → ReLU"""
    x = layers.Conv1D(
        filters, kernel_size,
        strides=strides,
        padding='same',
        use_bias=False,
        kernel_initializer='he_normal'
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x


# ──────────────────────────────────────────────────────────────────
# MODEL FACTORY — Standard CNN Only
# ──────────────────────────────────────────────────────────────────

def build_cnn(input_shape, num_classes):
    """
    Deep 1D CNN for vibration-based bearing fault classification.

    Args:
        input_shape  : (window_size, channels) — e.g. (1024, 1)
        num_classes  : number of fault/health categories
    """
    inputs = Input(shape=input_shape, name='vibration_input')
    x = inputs

    # Block 1: wide kernel to capture low-freq patterns
    x = conv_bn_relu(x, 64,  kernel_size=7)
    x = layers.MaxPooling1D(pool_size=4)(x)          # 1024 → 256

    # Block 2
    x = conv_bn_relu(x, 128, kernel_size=5)
    x = layers.MaxPooling1D(pool_size=4)(x)          # 256 → 64

    # Block 3
    x = conv_bn_relu(x, 256, kernel_size=3)
    x = layers.MaxPooling1D(pool_size=4)(x)          # 64 → 16

    # Block 4: no downsampling — refine features
    x = conv_bn_relu(x, 256, kernel_size=3)

    # ── HEAD ────────────────────────────────────────────────────────
    x = layers.GlobalAveragePooling1D()(x)               # fixed-size vector, no params

    x = layers.Dense(256, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, use_bias=False, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs, name='DeepCNN1D')
    return model


# Build the model
model = build_cnn(
    input_shape=(WINDOW_SIZE, 1),
    num_classes=NUM_CLASSES
)

model.summary()

## 5. Compile & Train

In [ ]:
# ── Compile ──────────────────────────────────────────────────────
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
]

# ── Train ─────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train_oh,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val_oh),
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

## 6. Evaluation

In [ ]:
# ── Training curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Test set evaluation ──────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print(f"\nWeighted F1 : {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Macro F1    : {f1_score(y_test, y_pred, average='macro'):.4f}")

In [ ]:
# ── Classification Report ────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_
))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()